In [14]:
using JuMP, HiGHS, LinearAlgebra

In [2]:
m = Model(HiGHS.Optimizer)
@variable(m, x₁ ≥ 0)
@variable(m, x₂ ≥ 0)
@objective(m, Max, 2x₁+4x₂)
@constraint(m, c1, 4x₁+3x₂ ≤ 120)
@constraint(m, c2, x₁+2x₂ ≤ 40)
@constraint(m, c3, x₂ ≤ 16)
print(m)


Max 2 x₁ + 4 x₂
Subject to
 c1 : 4 x₁ + 3 x₂ ≤ 120
 c2 : x₁ + 2 x₂ ≤ 40
 c3 : x₂ ≤ 16
 x₁ ≥ 0
 x₂ ≥ 0


In [3]:
set_silent(m)
optimize!(m)
is_solved_and_feasible(m)

true

In [4]:
value(x₁), value(x₂), objective_value(m)

(8.0, 16.0, 80.0)

In [5]:
dual_status(m)

FEASIBLE_POINT::ResultStatusCode = 1

In [6]:
dual(c1), dual(c2), dual(c3)

(0.0, -2.0, -0.0)

In [7]:
shadow_price(c1),shadow_price(c2), shadow_price(c3)

(-0.0, 2.0, 0.0)

In [8]:
Dict(
    xi => get_attribute(xi, MOI.VariableBasisStatus()) for
    xi in all_variables(m)
)

Dict{VariableRef, MathOptInterface.BasisStatusCode} with 2 entries:
  x₂ => BASIC
  x₁ => BASIC

In [9]:
MOI.get(m, MOI.ConstraintBasisStatus(), c1), MOI.get(m, MOI.ConstraintBasisStatus(), c2),  MOI.get(m, MOI.ConstraintBasisStatus(), c3)

(MathOptInterface.BASIC, MathOptInterface.NONBASIC, MathOptInterface.NONBASIC)

In [10]:
map(c-> name(c) => MOI.get(m, MOI.ConstraintBasisStatus(), c), all_constraints(m, include_variable_in_set_constraints = false))

3-element Vector{Pair{String, MathOptInterface.BasisStatusCode}}:
 "c1" => MathOptInterface.BASIC
 "c2" => MathOptInterface.NONBASIC
 "c3" => MathOptInterface.NONBASIC

In [11]:
reduced_cost(x₁), reduced_cost(x₂)

(-0.0, -0.0)

## $n$-queens

In [30]:
m2 = Model(HiGHS.Optimizer)
n = 4
P = [ i+j == n+1 ? 1 : 0 for i=1:n, j=1:n]
@variable(m2, x[1:n,1:n], Bin)
@constraint(m2, c1, mapslices(sum, x, dims = [1]) .<= ones(1,n))
@constraint(m2, c2, mapslices(sum, x, dims = [2]) .<= ones(n,1))
@constraint(m2, c3, [sum(diagm(i=>ones(n-abs(i))).*x) for i=-(n-2):(n-2)] .<= ones(2n-3))
@constraint(m2, c4, [sum(P*diagm(i=>ones(n-abs(i))).*x) for i=-(n-2):(n-2)] .<= ones(2n-3))
@objective(m2, Max, sum([rand(0.99:0.001:1.01)*x[i,j] for i=1:n, j=1:n]))
print(m2)

Max 1.006 x[1,1] + 0.99 x[2,1] + 0.997 x[3,1] + 1.004 x[4,1] + x[1,2] + 0.99 x[2,2] + 1.01 x[3,2] + 0.994 x[4,2] + 1.002 x[1,3] + 1.008 x[2,3] + 1.001 x[3,3] + 0.993 x[4,3] + 1.001 x[1,4] + 1.003 x[2,4] + 0.998 x[3,4] + 0.999 x[4,4]
Subject to
 c1 : x[1,1] + x[2,1] + x[3,1] + x[4,1] ≤ 1
 c1 : x[1,2] + x[2,2] + x[3,2] + x[4,2] ≤ 1
 c1 : x[1,3] + x[2,3] + x[3,3] + x[4,3] ≤ 1
 c1 : x[1,4] + x[2,4] + x[3,4] + x[4,4] ≤ 1
 c2 : x[1,1] + x[1,2] + x[1,3] + x[1,4] ≤ 1
 c2 : x[2,1] + x[2,2] + x[2,3] + x[2,4] ≤ 1
 c2 : x[3,1] + x[3,2] + x[3,3] + x[3,4] ≤ 1
 c2 : x[4,1] + x[4,2] + x[4,3] + x[4,4] ≤ 1
 c3 : x[3,1] + x[4,2] ≤ 1
 c3 : x[2,1] + x[3,2] + x[4,3] ≤ 1
 c3 : x[1,1] + x[2,2] + x[3,3] + x[4,4] ≤ 1
 c3 : x[1,2] + x[2,3] + x[3,4] ≤ 1
 c3 : x[1,3] + x[2,4] ≤ 1
 c4 : x[2,1] + x[1,2] ≤ 1
 c4 : x[3,1] + x[2,2] + x[1,3] ≤ 1
 c4 : x[4,1] + x[3,2] + x[2,3] + x[1,4] ≤ 1
 c4 : x[4,2] + x[3,3] + x[2,4] ≤ 1
 c4 : x[4,3] + x[3,4] ≤ 1
 x[1,1] binary
 x[2,1] binary
 x[3,1] binary
 x[4,1] binary
 x[1,2] bina

In [31]:
set_silent(m2)
optimize!(m2)
is_solved_and_feasible(m2)

true

In [32]:
round.(Int, value.(x))

4×4 Matrix{Int64}:
 0  1  0  0
 0  0  0  1
 1  0  0  0
 0  0  1  0

In [22]:
Dict(
    xi => get_attribute(xi, MOI.VariableBasisStatus()) for
    xi in all_variables(m2)
)

Dict{VariableRef, MathOptInterface.BasisStatusCode} with 16 entries:
  x[4,4] => NONBASIC
  x[1,3] => NONBASIC
  x[1,2] => NONBASIC
  x[3,3] => NONBASIC
  x[2,2] => NONBASIC
  x[1,1] => NONBASIC
  x[3,2] => NONBASIC
  x[4,2] => NONBASIC
  x[2,3] => NONBASIC
  x[2,1] => NONBASIC
  x[4,3] => NONBASIC
  x[3,1] => NONBASIC
  x[3,4] => NONBASIC
  x[4,1] => NONBASIC
  x[1,4] => NONBASIC
  x[2,4] => NONBASIC

In [25]:
map(c-> name(c) => MOI.get(m2, MOI.ConstraintBasisStatus(), c), all_constraints(m2, include_variable_in_set_constraints = false))

18-element Vector{Pair{String, MathOptInterface.BasisStatusCode}}:
 "c1" => MathOptInterface.BASIC
 "c1" => MathOptInterface.BASIC
 "c1" => MathOptInterface.BASIC
 "c1" => MathOptInterface.BASIC
 "c2" => MathOptInterface.BASIC
 "c2" => MathOptInterface.BASIC
 "c2" => MathOptInterface.BASIC
 "c2" => MathOptInterface.BASIC
 "c3" => MathOptInterface.BASIC
 "c3" => MathOptInterface.BASIC
 "c3" => MathOptInterface.BASIC
 "c3" => MathOptInterface.BASIC
 "c3" => MathOptInterface.BASIC
 "c4" => MathOptInterface.BASIC
 "c4" => MathOptInterface.BASIC
 "c4" => MathOptInterface.BASIC
 "c4" => MathOptInterface.BASIC
 "c4" => MathOptInterface.BASIC

In [29]:
dual.(UpperBoundRef.(x))

ErrorException: Variable x[1,1] does not have an upper bound.